In [1]:
!pip install pyspark --quiet
print('pyspark installation complete')

pyspark installation complete


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
spark=SparkSession.builder \
  .appName('Day4 bigdata sales') \
  .config('spark.sql.adaptive.enabled','true') \
  .getOrCreate()
print(f'spark version: {spark.version}')
print(f'spark session:ACTIVE')
print(f'application name: {spark.conf.get("spark.app.name")}')

spark version: 4.0.2
spark session:ACTIVE
application name: Day4 bigdata sales


In [3]:
from google.colab import files
uploaded = files.upload()

Saving large_sales_data.csv to large_sales_data.csv


In [5]:
df_bronze=spark.read \
.option('header','true') \
.option('inferSchema','true') \
.csv('large_sales_data.csv')
print('=== bronze layer -raw data ===')
print(f'rows: {df_bronze.count()}')
print(f'columns: {len(df_bronze.columns)}')
print(f'names:{df_bronze.columns}')
print()
df_bronze.printSchema()

=== bronze layer -raw data ===
rows: 5000
columns: 13
names:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [6]:
print('First 5 rows:')
df_bronze.show(5,truncate=False)

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [14]:
df_bronze.write \
  .mode('overwrite') \
  .parquet('sales_bronze.parquet')

print('Bronze Parquet saved: sales_bronze.parquet')

#compare file_sizes
import os

def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.isfile(path)
  total=0
  for dirpath,dirnames,filenames in os.walk(path):
    for f in filenames:
      total+=os.path.getsize(os.path.join(dirpath,f))
    return total/1024

csv_size=get_dir_size('large_sales_data.csv')
parquet_size=get_dir_size('sales_bronze.parquet')
reduction=(1-parquet_size/csv_size)*100
print(f'\ncsv file size: {csv_size:.1f} KB')
print(f'parquet file size: {parquet_size:.1f} KB')
print(f'reduction in size: {reduction:.1f}% smaller')
print(f'\nAt 1TB scale: CSV=1000GB -> Parquet={1000*(1-reduction/100):.0f} GB')

Bronze Parquet saved: sales_bronze.parquet

csv file size: 1.0 KB
parquet file size: 55.1 KB
reduction in size: -5409.8% smaller

At 1TB scale: CSV=1000GB -> Parquet=55098 GB


In [15]:
df_silver=df_bronze \
  .dropDuplicates() \
  .dropna(subset=['order_id','product','revenue'])

df_silver=df_silver.withColumn(
     'order_date',to_date(col('order_date'),'yyyy-MM-dd')
  )

df_silver=df_silver \
  .withColumn('order_year',year('order_date')) \
  .withColumn('order_month',month('order_date'))

df_silver=df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue')>40000,'High')
    .when(col('revenue')>10000,'Medium')
    .otherwise('Low')
)

print(f'Silver layer rows:{df_silver.count()}')
print(f'New columns added: order_year ,order_month,revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show()

Silver layer rows:5000
New columns added: order_year ,order_month,revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|  Keyboard|  13200|      2023|          2|          Medium|
|    Webcam|  17500|      2023|          1|          Medium|
|   Speaker|  58500|      2023|          4|            High|
|  Keyboard|   9600|      2023|         12|             Low|
|    Laptop| 180000|      2023|          8|            High|
|Headphones|  38500|      2023|          5|          Medium|
|    Webcam|  35000|      2023|         11|          Medium|
|    Laptop| 360000|      2023|          1|            High|
|    Tablet| 320000|      2023|          6|            High|
|    Laptop| 225000|      2023|          6|            High|
|     Mouse|   6400|      2023|          8|             Low|
|   Monitor| 132000|      2023|          7|            High|
| 

In [16]:
df_silver.write \
  .mode('overwrite') \
  .parquet('sales_silver.parquet')

print('Silver Parquet saved: sales_silver.parquet')
print(f'Silver size: {get_dir_size("sales_silver.parquet"):.1f} KB')

df_verify=spark.read.parquet('sales_silver.parquet')
print('\n===Verify Silver layer===')
print(f'Read back rows:{df_verify.count()} (should match silver count)')
df_verify.printSchema()

Silver Parquet saved: sales_silver.parquet
Silver size: 59.8 KB

===Verify Silver layer===
Read back rows:5000 (should match silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)

